In [2]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import pandas as pd
import csv
import os
import joblib

# load the models from the file
models = joblib.load('running_analysis_models.joblib')
    
# feature and target definition
X_features = [
    'cadence_value (spm)', 'gct_value (ms)', 'vert_osc_value (%)', 
    'knee_strike_angle (deg)', 'knee_push_angle (deg)', 'elbow_angle_val (deg)', 
    'leg_split_val (deg)', 'trunk_lean_value (deg)',
    'n_ankle_x', 'n_ankle_y', 'n_knee_x', 'n_knee_y', 
    'n_foot_stretch', 'n_heel_toe_slope', 'n_knee_elevation', 
    'n_shoulder_lean', 'n_elbow_x'
]

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# angle calculation
def calculate_angle(point_a, point_b, point_c):
    point_a, point_b, point_c = np.array(point_a), np.array(point_b), np.array(point_c)
    vector_ba = point_a - point_b
    vector_bc = point_c - point_b
    cosine_angle = np.dot(vector_ba, vector_bc) / (np.linalg.norm(vector_ba) * np.linalg.norm(vector_bc))
    return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))

# additional feature preparation
def prepare_ml_features(lm, side, frame_width, frame_height):
    if side == "Right":
        hip, knee, ankle = 24, 26, 28
        shoulder, elbow, wrist = 12, 14, 16
        heel, toe = 30, 32
    else:
        hip, knee, ankle = 23, 25, 27
        shoulder, elbow, wrist = 11, 13, 15
        heel, toe = 29, 31

    mid_hip_x = (lm[24].x + lm[23].x) / 2
    mid_hip_y = (lm[24].y + lm[23].y) / 2
    mid_shoulder_x = (lm[12].x + lm[11].x) / 2
    mid_shoulder_y = (lm[12].y + lm[11].y) / 2

    torso_dist = np.sqrt((mid_shoulder_x - mid_hip_x)**2 + (mid_shoulder_y - mid_hip_y)**2)
    if torso_dist == 0: torso_dist = 1

    def norm_x(l_idx): return (lm[l_idx].x - mid_hip_x) / torso_dist
    def norm_y(l_idx): return (lm[l_idx].y - mid_hip_y) / torso_dist

    features = {
        'n_ankle_x': round(norm_x(ankle), 4),
        'n_ankle_y': round(norm_y(ankle), 4),
        'n_knee_x': round(norm_x(knee), 4),
        'n_knee_y': round(norm_y(knee), 4),
        'n_foot_stretch': round(norm_x(ankle) - norm_x(hip), 4),
        'n_heel_toe_slope': round(lm[heel].y - lm[toe].y, 4),
        'n_knee_elevation': round(norm_y(knee), 4),
        'n_shoulder_lean': round(norm_x(shoulder), 4),
        'n_elbow_x': round(norm_x(elbow), 4)
    }
    return features

# main video analysis function
def analyze_video(video_path, trained_models=None, features_list=None):
    video_capture = cv2.VideoCapture(video_path)
    frames_per_second = video_capture.get(cv2.CAP_PROP_FPS)

    # thresholds and parameters
    push_off_threshold = 0.08
    min_swing_frames = frames_per_second * 0.22 
    min_contact_frames = 3
    gct_timeout = 1.0

    # state and history initialization
    all_step_metrics_storage = []
    history_window_size = 10
    right_knee_angle_history = deque(maxlen=history_window_size)
    left_knee_angle_history = deque(maxlen=history_window_size)
    hip_height_history = deque(maxlen=50)
    cadence_history = deque(maxlen=5)
    gct_filter_history = deque(maxlen=5)

    current_max_split = 0
    leg_is_on_ground = {"Right": False, "Left": False}
    ankle_y_at_contact = {"Right": None, "Left": None}
    last_ankle_y = {"Right": None, "Left": None}
    last_leg_that_landed = None
    last_strike_frame_index = 0
    previous_strike_frame = None
    avg_cadence = 0
    right_step_count = 0
    left_step_count = 0
    current_status_event = ""
    latest_ai_results = {}
    last_display_frame = None 

    # pose estimation setup
    with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose_analyzer:
        while video_capture.isOpened():
            ret, frame = video_capture.read()
            if not ret: break

            overlay_layer = frame.copy()
            display_frame = frame.copy()
            frame_height, frame_width, _ = frame.shape
            results = pose_analyzer.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                def get_pixel_point(idx): return np.array([int(landmarks[idx].x * frame_width), int(landmarks[idx].y * frame_height)])

                # keypoint extraction
                r_hip, r_knee, r_ankle = get_pixel_point(24), get_pixel_point(26), get_pixel_point(28)
                l_hip, l_knee, l_ankle = get_pixel_point(23), get_pixel_point(25), get_pixel_point(27)
                r_shld, r_elb, r_wrst = get_pixel_point(12), get_pixel_point(14), get_pixel_point(16)
                l_shld, l_elb, l_wrst = get_pixel_point(11), get_pixel_point(13), get_pixel_point(15)

                # angle calculations
                r_knee_ang = calculate_angle(r_hip, r_knee, r_ankle)
                l_knee_ang = calculate_angle(l_hip, l_knee, l_ankle)
                r_elb_ang = calculate_angle(r_shld, r_elb, r_wrst)
                l_elb_ang = calculate_angle(l_shld, l_elb, l_wrst)

                # trunk lean calculation
                trunk_vector = np.array([landmarks[12].x - landmarks[24].x, landmarks[12].y - landmarks[24].y])
                trunk_angle = 180 - np.degrees(np.arctan2(np.abs(trunk_vector[0]), np.abs(trunk_vector[1])))

                # vertical oscillation calculation
                torso_h = np.abs(landmarks[24].y - landmarks[12].y) 
                mid_h_y = (landmarks[24].y + landmarks[23].y) / 2
                hip_height_history.append(mid_h_y)
                v_osc = ((max(hip_height_history) - min(hip_height_history)) / torso_h) * 100 if torso_h > 0 else 0

                # leg split calculation
                v_r, v_l = (r_knee - r_hip), (l_knee - l_hip)
                split_angle = np.degrees(np.arccos(np.clip(np.dot(v_r/np.linalg.norm(v_r), v_l/np.linalg.norm(v_l)), -1.0, 1.0)))
                if split_angle > current_max_split: current_max_split = split_angle

                # draw skeletons and torso overlays
                torso_pts = np.array([r_shld, l_shld, l_hip, r_hip], np.int32)
                cv2.fillPoly(overlay_layer, [torso_pts], (0, 255, 0)) 
                cv2.addWeighted(overlay_layer, 0.15, display_frame, 0.85, 0, display_frame)
                cv2.polylines(display_frame, [torso_pts], True, (255, 255, 255), 2)
                cv2.line(display_frame, tuple(r_shld), tuple(l_hip), (255, 255, 255), 1)
                cv2.line(display_frame, tuple(l_shld), tuple(r_hip), (255, 255, 255), 1)
                cv2.line(display_frame, tuple(r_ankle), tuple(l_ankle), (255, 0, 255), 2)

                # draw leg and arm lines
                for h, k, a, c in [(r_hip, r_knee, r_ankle, (0, 255, 0)), (l_hip, l_knee, l_ankle, (0, 255, 255))]:
                    cv2.line(display_frame, tuple(h), tuple(k), c, 3)
                    cv2.line(display_frame, tuple(k), tuple(a), c, 3)
                for s, e, w in [(r_shld, r_elb, r_wrst), (l_shld, l_elb, l_wrst)]:
                    cv2.line(display_frame, tuple(s), tuple(e), (255, 165, 0), 3)
                    cv2.line(display_frame, tuple(e), tuple(w), (255, 165, 0), 3)

                # add angle labels
                for j, ang, c in [(r_knee, r_knee_ang, (0,255,0)), (l_knee, l_knee_ang, (0,255,255)), (r_elb, r_elb_ang, (255,165,0))]:
                    cv2.putText(display_frame, f"{int(ang)}", tuple(j + [10, -10]), 1, 1.2, c, 2)

                right_knee_angle_history.append(r_knee_ang)
                left_knee_angle_history.append(l_knee_ang)
                current_frame = video_capture.get(cv2.CAP_PROP_POS_FRAMES)

                # foot strike detection and ML trigger
                if len(right_knee_angle_history) == history_window_size:
                    mid_idx = history_window_size // 2
                    lockout = frames_per_second * 0.18 if avg_cadence > 190 else min_swing_frames

                    for side, history, ankle_y, e_ang in [("Right", right_knee_angle_history, landmarks[28].y, r_elb_ang), ("Left", left_knee_angle_history, landmarks[27].y, l_elb_ang)]:
                        if (last_leg_that_landed != side and history[mid_idx] > 150 and history[mid_idx] == max(history) and (current_frame - last_strike_frame_index) > lockout):
                            other_side = "Left" if side == "Right" else "Right"
                            if leg_is_on_ground[other_side]:
                                for rec in reversed(all_step_metrics_storage):
                                    if rec['side'] == other_side and not rec['done']:
                                        raw_val = ((current_frame - rec['start_frame']) / frames_per_second) * 1000
                                        gct_filter_history.append(raw_val)
                                        rec['gct'] = sum(gct_filter_history) / len(gct_filter_history)
                                        rec['done'] = True
                                        
                                        # run AI prediction on finished step
                                        if trained_models:
                                            mapping = {'cadence':'cadence_value (spm)', 'gct':'gct_value (ms)', 'v_osc':'vert_osc_value (%)', 'strike_knee':'knee_strike_angle (deg)', 'push_knee':'knee_push_angle (deg)', 'elbow':'elbow_angle_val (deg)', 'split':'leg_split_val (deg)', 'trunk':'trunk_lean_value (deg)'}
                                            step_df = pd.DataFrame([rec]).rename(columns=mapping)
                                            for target, model in trained_models.items():
                                                latest_ai_results[target] = model.predict(step_df[features_list])[0]
                                        leg_is_on_ground[other_side] = False
                                        break

                            leg_is_on_ground[side], ankle_y_at_contact[side], last_leg_that_landed = True, ankle_y, side
                            current_status_event = f"{side.upper()} STRIKE"
                            
                            # update cadence
                            if previous_strike_frame is not None:
                                cadence_history.append((60 * frames_per_second) / (current_frame - previous_strike_frame))
                                avg_cadence = sum(cadence_history) / len(cadence_history)
                                
                            previous_strike_frame, last_strike_frame_index = current_frame, current_frame
                            if side == "Right": right_step_count += 1
                            else: left_step_count += 1
                            
                            # store new step metrics
                            ml_data = prepare_ml_features(landmarks, side, frame_width, frame_height)
                            all_step_metrics_storage.append({'side': side, 'strike_knee': history[mid_idx], 'split': current_max_split, 'trunk': trunk_angle, 'cadence': avg_cadence, 'v_osc': v_osc, 'elbow': e_ang, 'start_frame': current_frame, 'push_knee': history[mid_idx], **ml_data, 'done': False, 'gct': 0})
                            current_max_split = 0

                # push-off detection logic
                for side, current_y in [("Right", landmarks[28].y), ("Left", landmarks[27].y)]:
                    if leg_is_on_ground[side]:
                        for rec in reversed(all_step_metrics_storage):
                            if rec['side'] == side and not rec['done']:
                                c_ang = r_knee_ang if side == "Right" else l_knee_ang
                                if c_ang > rec['push_knee']: rec['push_knee'] = c_ang
                                frames_ground = current_frame - rec['start_frame']
                                moving_up = (current_y < last_ankle_y[side]) if last_ankle_y[side] is not None else False
                                if (ankle_y_at_contact[side] - current_y) > push_off_threshold and frames_ground >= min_contact_frames and moving_up:
                                    raw_val = (frames_ground / frames_per_second) * 1000
                                    gct_filter_history.append(raw_val)
                                    rec['gct'] = sum(gct_filter_history) / len(gct_filter_history)
                                    rec['done'] = True
                                    
                                    # run AI prediction on push-off
                                    if trained_models:
                                        mapping = {'cadence':'cadence_value (spm)', 'gct':'gct_value (ms)', 'v_osc':'vert_osc_value (%)', 'strike_knee':'knee_strike_angle (deg)', 'push_knee':'knee_push_angle (deg)', 'elbow':'elbow_angle_val (deg)', 'split':'leg_split_val (deg)', 'trunk':'trunk_lean_value (deg)'}
                                        step_df = pd.DataFrame([rec]).rename(columns=mapping)
                                        for target, model in trained_models.items():
                                            latest_ai_results[target] = model.predict(step_df[features_list])[0]
                                            
                                    leg_is_on_ground[side] = False
                                    current_status_event = f"{side.upper()} PUSH-OFF"
                                    break
                        last_ankle_y[side] = current_y

                # top-left telemetry UI
                cv2.rectangle(display_frame, (0,0), (280, 100), (20,20,20), -1)
                cv2.putText(display_frame, f"STEPS: {right_step_count + left_step_count}", (15, 35), 1, 1.8, (255,255,255), 2)
                cv2.putText(display_frame, f"CADENCE: {int(avg_cadence)}", (15, 65), 1, 1.2, (0, 255, 0), 2)
                cv2.putText(display_frame, f"STATUS: {current_status_event}", (15, 95), 1, 1.0, (0, 255, 255), 1)

                # top-right live AI analysis UI
                if latest_ai_results:
                    bw, start_y, sx = 250, 35, frame_width - 260
                    cv2.rectangle(display_frame, (sx, 10), (frame_width - 10, start_y + 160), (20, 20, 20), -1)
                    cv2.putText(display_frame, "AI ANALYSIS", (sx + 15, start_y), 1, 1.2, (255, 255, 255), 2)
                    for i, (metric, score) in enumerate(latest_ai_results.items()):
                        y_p = start_y + 30 + (i * 18)
                        val = int(score)
                        color = (0, 255, 0) if val >= 3 else (0, 255, 255) if val == 2 else (0, 0, 255)
                        cv2.putText(display_frame, f"{metric.replace('_score', '').upper()}:", (sx + 15, y_p), 1, 0.8, (200, 200, 200), 1)
                        cv2.putText(display_frame, f"{val}", (sx + bw - 30, y_p), 1, 1.0, color, 2)

                # bottom contact status UI
                for side_ui, pos, state in [("LEFT", (10, frame_height-20), leg_is_on_ground["Left"]), ("RIGHT", (frame_width-130, frame_height-20), leg_is_on_ground["Right"])]:
                    color = (0, 255, 0) if state else (0, 0, 255)
                    cv2.rectangle(display_frame, (pos[0]-10, pos[1]-40), (pos[0]+120, pos[1]+10), (0,0,0), -1)
                    cv2.putText(display_frame, side_ui, (pos[0], pos[1]-20), 1, 1.2, color, 2)
                    cv2.putText(display_frame, "CONTACT" if state else "FLIGHT", (pos[0], pos[1]), 1, 0.9, (255,255,255), 1)

                cv2.imshow('Running analysis - side view', display_frame)
                last_display_frame = display_frame.copy()
                if cv2.waitKey(1) & 0xFF == ord('q'): break

    # session summary screen logic
    if last_display_frame is not None and trained_models:
        test_df = pd.DataFrame(all_step_metrics_storage)
        test_df = test_df[test_df['done'] == True].copy()
        if not test_df.empty:
            mapping = {'cadence':'cadence_value (spm)', 'gct':'gct_value (ms)', 'v_osc':'vert_osc_value (%)', 'strike_knee':'knee_strike_angle (deg)', 'push_knee':'knee_push_angle (deg)', 'elbow':'elbow_angle_val (deg)', 'split':'leg_split_val (deg)', 'trunk':'trunk_lean_value (deg)'}
            model_df = test_df.rename(columns=mapping)
            
            # create dim overlay
            overlay = last_display_frame.copy()
            cv2.rectangle(overlay, (0, 0), (frame_width, frame_height), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.7, last_display_frame, 0.3, 0, last_display_frame)
            
            # draw result box
            cx, cy = frame_width // 2, frame_height // 2
            cv2.rectangle(last_display_frame, (cx - 250, cy - 225), (cx + 250, cy + 225), (30, 30, 30), -1)
            cv2.rectangle(last_display_frame, (cx - 250, cy - 225), (cx + 250, cy + 225), (255, 255, 255), 2)
            cv2.putText(last_display_frame, "SESSION SUMMARY", (cx - 150, cy - 180), 1, 2.0, (255, 255, 255), 2)

            # display average AI scores
            for i, (target, model) in enumerate(trained_models.items()):
                avg_score = int(round(np.mean(model.predict(model_df[features_list]))))
                color = (0, 255, 0) if avg_score >= 3 else (0, 255, 255) if avg_score == 2 else (0, 0, 255)
                y_p = cy - 100 + (i * 35)
                cv2.putText(last_display_frame, f"{target.replace('_score','').upper()}:", (cx - 220, y_p), 1, 1.2, (200, 200, 200), 1)
                cv2.putText(last_display_frame, f"{avg_score}", (cx + 180, y_p), 1, 1.5, color, 2)

            cv2.imshow('Running analysis - side view', last_display_frame)
            cv2.waitKey(1)

    video_capture.release()
    cv2.destroyAllWindows()
    return all_step_metrics_storage

# run test on a video
video_name = './Videos/Video.mov'
results = analyze_video(video_name, models, X_features)

I0000 00:00:1769732206.415750 1072728 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769732206.478575 1080281 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769732206.490836 1080284 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
